In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "api key"

In [19]:
%pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Step1 : Document Ingestion

In [20]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled

video_id = "aircAruvnKk"

try:
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(video_id, languages=["en"])
    
    # Use chunk.text instead of chunk["text"]
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

This is a 3. It's sloppily written and rendered at an extremely low resolution of 28x28 pixels, but your brain has no trouble recognizing it as a 3. And I want you to take a moment to appreciate how crazy it is that brains can do this so effortlessly. I mean, this, this and this are also recognizable as 3s, even though the specific values of each pixel is very different from one image to the next. The particular light-sensitive cells in your eye that are firing when you see this 3 are very different from the ones firing when you see this 3. But something in that crazy-smart visual cortex of yours resolves these as representing the same idea, while at the same time recognizing other images as their own distinct ideas. But if I told you, hey, sit down and write for me a program that takes in a grid of 28x28 pixels like this and outputs a single number between 0 and 10, telling you what it thinks the digit is, well the task goes from comically trivial to dauntingly difficult. Unless you'v

[
    {'text': 'Hello everyone', 'start': 0.12, 'duration': 1.50},
    {'text': 'Welcome to the tutorial', 'start': 1.62, 'duration': 2.10}
]


Text Splitting 

In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Text splitting configuration[cite: 1]
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])
len(chunks)


23

In [7]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Initialize Gemini embeddings (using the standard gemini-embedding model)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Build the FAISS vector store using the chunks and Gemini embeddings
vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.index_to_docstore_id

{0: 'eb88f612-ff1b-47a6-a2bc-e3a976ab7dac',
 1: 'fd13fe31-0602-466f-a8ea-e012fcbd53f0',
 2: '583b0f5c-03ae-4eb5-9d2b-b0dd29123e39',
 3: '6161c204-53d8-4f30-97d7-06372ca2be62',
 4: '3fb87128-a743-434c-8088-744fd2664e5e',
 5: '000e92ed-ac3b-48e0-9775-13d376271a80',
 6: '6946975a-eb3c-4a67-bdb4-9a3acfb02cd2',
 7: 'd77241e8-675a-4dbd-b602-0b5407177df2',
 8: 'c96487ac-afb5-4e4c-b2fe-00887b6cf752',
 9: 'c3ecaea3-1d72-4284-b553-7333104d1122',
 10: 'aee3866e-d514-4b9f-88ce-4e49b4fe735e',
 11: '575b1763-b695-423b-bb84-cf36dcb0c147',
 12: '7021662c-5e2c-4dd1-8c50-158575fd739f',
 13: '9eabd4fa-d9d5-4d38-b385-54fcc077089e',
 14: '30da643c-c38c-41fd-887e-f7183bf40d8e',
 15: '72a2f01f-25dd-4521-bc6b-e27a760133fb',
 16: 'db0ecb16-4e1a-447f-a4cc-cc30bb767b1d',
 17: '75b84150-61b8-44f2-8249-e3ec2afbdc69',
 18: '7ed487ea-26b0-4434-8dc4-df5d453e33d0',
 19: '3a300a54-84ff-4e5f-acd7-84ae6b9766a7',
 20: 'e403d464-e186-4d42-bdb4-22da263c5e57',
 21: 'cbfc1915-ea48-42e3-92ae-316285d3be09',
 22: '1b340d28-c9dc-

Retrival

In [8]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever.invoke('what is neural network')

[Document(id='6161c204-53d8-4f30-97d7-06372ca2be62', metadata={}, page_content="the more powerful modern variants, and trust me it still has plenty of complexity for us to wrap our minds around. But even in this simplest form it can learn to recognize handwritten digits, which is a pretty cool thing for a computer to be able to do. And at the same time you'll see how it does fall short of a couple hopes that we might have for it. As the name suggests neural networks are inspired by the brain, but let's break that down. What are the neurons, and in what sense are they linked together? Right now when I say neuron all I want you to think about is a thing that holds a number, specifically a number between 0 and 1. It's really not more than that. For example the network starts with a bunch of neurons corresponding to each of the 28x28 pixels of the input image, which is 784 neurons in total. Each one of these holds a number that represents the grayscale value of the corresponding pixel, ran

In [9]:
Augmentation

NameError: name 'Augmentation' is not defined

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
# Initialize the Gemini model (requires the langchain-google-genai package and a GOOGLE_API_KEY)
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.2)

prompt = PromptTemplate(
    template="""
    Act as an assisting AI.
    Base your response entirely on the transcript context supplied below.
    State that you do not know the answer if the provided context is not sufficient.

    {context}
    Question: {question}
    """,
    input_variables=['context', 'question']
)

question = "Does the video mention the topic of neural network? If so, detail what is said."
retrieved_docs = retriever.invoke(question)
retrieved_docs

[Document(id='583b0f5c-03ae-4eb5-9d2b-b0dd29123e39', metadata={}, page_content="of that, and the following one is going to tackle learning. What we're going to do is put together a neural network that can learn to recognize handwritten digits. This is a somewhat classic example for introducing the topic, and I'm happy to stick with the status quo here, because at the end of the two videos I want to point you to a couple good resources where you can learn more, and where you can download the code that does this and play with it on your own computer. There are many many variants of neural networks, and in recent years there's been sort of a boom in research towards these variants, but in these two introductory videos you and I are just going to look at the simplest plain vanilla form with no added frills. This is kind of a necessary prerequisite for understanding any of the more powerful modern variants, and trust me it still has plenty of complexity for us to wrap our minds around. But 

In [24]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text="\n    Act as an assisting AI.\n    Base your response entirely on the transcript context supplied below.\n    State that you do not know the answer if the provided context is not sufficient.\n\n    of that, and the following one is going to tackle learning. What we're going to do is put together a neural network that can learn to recognize handwritten digits. This is a somewhat classic example for introducing the topic, and I'm happy to stick with the status quo here, because at the end of the two videos I want to point you to a couple good resources where you can learn more, and where you can download the code that does this and play with it on your own computer. There are many many variants of neural networks, and in recent years there's been sort of a boom in research towards these variants, but in these two introductory videos you and I are just going to look at the simplest plain vanilla form with no added frills. This is kind of a necessary prerequisite fo

Generation

In [16]:
answer = llm.invoke(final_prompt)
print(answer.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': 'Yes, the video mentions the topic of neural networks. Based on the provided context, here are the details of what is said about them:\n\n* **Definition and Goal:** The video aims to show what a neural network actually is (assuming no background) and visualize its operations as a piece of math rather than just a buzzword. It explains the structure of a neural network and what it means for a network to "learn."\n* **Application (Handwritten Digit Recognition):** The video demonstrates a neural network that can learn to recognize handwritten digits. It takes an input (a grid of 28x28 pixels) and outputs a single number between 0 and 10 representing what it thinks the digit is. \n* **Simplest Form vs. Variants:** While there are many variants of neural networks and a recent boom in research, the video focuses on the simplest, "plain vanilla" form with no added frills. Understanding this simple form is described as a necessary prerequisite for understanding more p